# 3.3_tool_result_to_model.ipynb

In [1]:
# User Input
#   ↓
# LLM 决定调用哪些 tools
#   ↓
# Python 执行 tools
#   ↓
# 把 tool result 作为 messages 回传给 LLM
#   ↓
# LLM 基于工具结果生成最终自然语言答案

## 导入依赖 + 初始化 Client

In [2]:
import sys
print(sys.executable)

d:\Code\swe_agent_jom\.venv\Scripts\python.exe


In [3]:
from __future__ import annotations

import json
import os
from datetime import date
from typing import Any, Callable, cast

from dotenv import load_dotenv
from openai import OpenAI


# =========================
# 1. 加载环境变量
# =========================
# 你的 .env 里面建议写：
# DEEPSEEK_API_KEY=你的key
load_dotenv()

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")

if not deepseek_api_key:
    raise ValueError("没有找到 DEEPSEEK_API_KEY，请检查 .env 文件")


# =========================
# 2. 初始化 OpenAI-compatible Client
# =========================
client = OpenAI(
    api_key=deepseek_api_key,
    base_url="https://api.deepseek.com",
)

MODEL = "deepseek-v4-flash"

## 准备一个生日数据表

In [4]:
# =========================
# 生日数据表
# =========================
# 格式统一用 MM-DD，方便按日期匹配。
birthday_table: dict[str, str] = {
    "Jack": "03-18",
    "Alice": "07-02",
    "Bob": "11-09",
    "Tom": "01-25",
}


# 学习演示用：
# 为了稳定看到效果，先固定“今天”为 2026-07-02。
# 如果你想使用真实日期，把 DEMO_TODAY 改成 None。
DEMO_TODAY: str | None = "2026-07-02"

## 实现真正的 Python 工具函数

In [5]:
def get_current_date() -> dict[str, str]:
    """
    获取当前日期。

    为了学习演示，我们允许用 DEMO_TODAY 固定今天。
    实际项目中可以直接使用 date.today().isoformat()。
    """
    today = DEMO_TODAY or date.today().isoformat()

    return {
        "today": today,
    }


def get_birthdays_by_date(target_date: str) -> dict[str, Any]:
    """
    根据指定日期查询当天生日的人。

    参数:
        target_date: 日期字符串，格式建议为 YYYY-MM-DD

    返回:
        {
            "target_date": "2026-07-02",
            "matched_people": ["Alice"]
        }
    """
    month_day = _extract_month_day(target_date)

    matched_people = [
        name
        for name, birthday in birthday_table.items()
        if birthday == month_day
    ]

    return {
        "target_date": target_date,
        "month_day": month_day,
        "matched_people": matched_people,
    }


def generate_basic_greeting(name: str, tone: str = "warm") -> dict[str, str]:
    """
    给某个人生成一句简单生日祝福。

    注意：
    这个函数故意写得很简单。
    目的不是展示祝福语有多高级，而是展示：
    模型可以在拿到第一个工具结果后，继续调用第二个工具。
    """
    if tone == "formal":
        greeting = f"祝{name}生日快乐，愿你新的一岁顺利安康，事业更进一步。"
    elif tone == "friendly":
        greeting = f"{name}，生日快乐！祝你今天开心，接下来一年好运不断！"
    else:
        greeting = f"祝{name}生日快乐，愿你新的一岁平安、顺利、开心。"

    return {
        "name": name,
        "tone": tone,
        "greeting": greeting,
    }


def _extract_month_day(target_date: str) -> str:
    """
    从 YYYY-MM-DD 中提取 MM-DD。

    这个函数是内部辅助函数，不暴露给模型。
    """
    parts = target_date.split("-")

    if len(parts) != 3:
        raise ValueError("target_date 必须是 YYYY-MM-DD 格式，例如 2026-07-02")

    month = parts[1]
    day = parts[2]

    return f"{month}-{day}"

## 定义 Tool Schema

In [6]:
tools: list[dict[str, Any]] = [
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "获取当前日期。当用户说“今天”但没有给出具体日期时，应该先调用这个工具。",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_birthdays_by_date",
            "description": "根据指定日期查询当天生日的人。必须传入 YYYY-MM-DD 格式的日期。",
            "parameters": {
                "type": "object",
                "properties": {
                    "target_date": {
                        "type": "string",
                        "description": "目标日期，格式为 YYYY-MM-DD，例如 2026-07-02",
                    },
                },
                "required": ["target_date"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "generate_basic_greeting",
            "description": "为指定的人生成一句生日祝福语。",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "需要生成祝福语的人名",
                    },
                    "tone": {
                        "type": "string",
                        "description": "祝福语语气，可选 warm、friendly、formal",
                        "enum": ["warm", "friendly", "formal"],
                    },
                },
                "required": ["name"],
            },
        },
    },
]

## 建立 Function Dispatch 工具注册表

In [7]:
# 工具函数的统一类型：
# key 是参数名，value 是参数值。
ToolFunction = Callable[..., Any]


tool_registry: dict[str, ToolFunction] = {
    "get_current_date": get_current_date,
    "get_birthdays_by_date": get_birthdays_by_date,
    "generate_basic_greeting": generate_basic_greeting,
}

## JSON 工具函数

In [8]:
def to_json_text(data: Any) -> str:
    """
    把 Python 对象转成 JSON 字符串。

    ensure_ascii=False:
        保证中文不会变成 Unicode 转义。
    """
    return json.dumps(data, ensure_ascii=False)


def parse_json_arguments(arguments: str) -> dict[str, Any]:
    """
    解析模型返回的 tool arguments。

    模型返回的 arguments 通常是 JSON 字符串，例如：
        '{"target_date": "2026-07-02"}'
    """
    if not arguments:
        return {}

    parsed = json.loads(arguments)

    if not isinstance(parsed, dict):
        raise ValueError("tool arguments 必须是 JSON object")

    return parsed

## 封装模型调用

In [9]:
def call_model(messages: list[dict[str, Any]]) -> Any:
    """
    调用模型，返回 assistant message。

    注意：
    这里使用 cast(Any, ...)
    是为了减少 OpenAI SDK 类型检查和普通 dict messages 之间的冲突。
    这在 notebook 学习阶段比较实用。
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=cast(Any, messages),
        tools=cast(Any, tools),
        tool_choice="auto",
    )

    return response.choices[0].message

## 执行单个 Tool Call

In [10]:
def execute_tool_call(tool_call: Any) -> dict[str, Any]:
    """
    执行模型提出的一个 tool call，并构造 role='tool' 的消息。

    输入:
        tool_call:
            模型返回的工具调用对象，里面包含：
            - tool_call.id
            - tool_call.function.name
            - tool_call.function.arguments

    输出:
        一条可以 append 到 messages 里的 tool message：
        {
            "role": "tool",
            "tool_call_id": "...",
            "content": "..."
        }
    """
    function_name = tool_call.function.name
    raw_arguments = tool_call.function.arguments

    try:
        arguments = parse_json_arguments(raw_arguments)

        if function_name not in tool_registry:
            result = {
                "ok": False,
                "error": f"Unknown tool: {function_name}",
            }
        else:
            tool_function = tool_registry[function_name]

            # 这里是真正执行 Python 函数的地方
            data = tool_function(**arguments)

            result = {
                "ok": True,
                "tool_name": function_name,
                "arguments": arguments,
                "data": data,
            }

    except Exception as error:
        # 不要让工具执行异常直接打断整个 Agent 流程。
        # 更好的方式是把错误也作为 tool result 回传给模型。
        result = {
            "ok": False,
            "tool_name": function_name,
            "error_type": type(error).__name__,
            "error_message": str(error),
        }

    tool_message = {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": to_json_text(result),
    }

    return tool_message

## 执行一轮模型回复里的所有 Tool Calls

In [11]:
def execute_all_tool_calls(assistant_message: Any) -> list[dict[str, Any]]:
    """
    执行 assistant message 里面的所有 tool calls。

    一个 assistant message 里面可能包含多个 tool call。
    例如模型可能一次性要求：
        1. 查询 Alice 生日
        2. 查询 Bob 生日
    """
    tool_calls = assistant_message.tool_calls or []

    tool_messages: list[dict[str, Any]] = []

    for tool_call in tool_calls:
        tool_message = execute_tool_call(tool_call)
        tool_messages.append(tool_message)

    return tool_messages

## 把 assistant message 转成 dict 后放进 messages

In [12]:
def append_assistant_message(
    messages: list[dict[str, Any]],
    assistant_message: Any,
) -> None:
    """
    把模型返回的 assistant message 追加到 messages。

    model_dump(exclude_none=True):
        把 SDK 对象转成普通 dict，并去掉 None 字段。
    """
    messages.append(
        assistant_message.model_dump(exclude_none=True)
    )

## 运行 Tool Result 回传闭环

In [13]:
def run_tool_result_demo(user_input: str, max_tool_rounds: int = 5) -> str:
    """
    运行一个完整的 tool result 回传模型流程。

    参数:
        user_input:
            用户输入

        max_tool_rounds:
            最多允许几轮工具调用。
            这是安全保护，避免模型一直调用工具停不下来。

    返回:
        模型最终自然语言回答。
    """
    messages: list[dict[str, Any]] = [
        {
            "role": "system",
            "content": (
                "你是一个生日助手。"
                "当用户询问今天或某天谁生日时，不要凭空猜测。"
                "你必须使用工具查询日期和生日数据。"
                "如果查到有人生日，并且用户需要祝福语，你应该调用祝福语工具。"
                "最后用中文给用户一个清晰、自然的回答。"
            ),
        },
        {
            "role": "user",
            "content": user_input,
        },
    ]

    for round_index in range(max_tool_rounds):
        print(f"\n========== Model Round {round_index + 1} ==========")

        assistant_message = call_model(messages)
        append_assistant_message(messages, assistant_message)

        print("Assistant message:")
        print(assistant_message)

        # 如果模型没有继续调用工具，说明它已经给出最终回答
        if not assistant_message.tool_calls:
            final_answer = assistant_message.content or ""
            return final_answer

        # 如果模型调用了工具，Python 执行工具
        tool_messages = execute_all_tool_calls(assistant_message)

        # 把每个工具结果回传进 messages
        for tool_message in tool_messages:
            print("\nTool result message:")
            print(tool_message)

            messages.append(tool_message)

    raise RuntimeError("超过 max_tool_rounds，模型仍然没有给出最终回答")

## 测试运行

In [14]:
answer = run_tool_result_demo(
    "今天有谁生日？如果有人生日，帮我生成一句友好一点的祝福语。"
)

print("\n========== Final Answer ==========")
print(answer)


========== Model Round 1 ==========
Assistant message:
ChatCompletionMessage(content='好的！我先查一下今天的日期。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_3ytAzjVbshLIYY4vQiUk1402', function=Function(arguments='{}', name='get_current_date'), type='function', index=0)], reasoning_content='用户想知道今天谁生日，但我不知道今天的日期。所以我需要先获取当前日期。让我调用 get_current_date 工具。')

Tool result message:
{'role': 'tool', 'tool_call_id': 'call_00_3ytAzjVbshLIYY4vQiUk1402', 'content': '{"ok": true, "tool_name": "get_current_date", "arguments": {}, "data": {"today": "2026-07-02"}}'}

========== Model Round 2 ==========
Assistant message:
ChatCompletionMessage(content='好的，今天是 **2026年7月2日**，我来查查今天过生日的人。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_LNCKMzh5xwDrau48Uo2U7662', function=Function(arguments='{"target_date": "2026-07-02"}',

## END